In [1]:
import pandas as pd
#importing pandas

In [5]:
#reading datasets and assigning to appropriate variables
data = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
test_ids = test["PassengerId"]

#<=========================This is a Function Start===============================>#
#<================================================================================>#
def clean(input_data):
    input_data = input_data.drop(["Ticket", "PassengerId", "Name", "Cabin"], axis=1)
    #dropping these fields cuz these are not related to surviving, like the name
    #axis=0 → Operate along rows (default) axis=1 → Operate along columns
    
    cols = ["SibSp", "Parch", "Fare", "Age"]
    #sibsp - sibling / spouse , Parch = Number of Parents/Children Aboard
    
    #taking specific columns for loop
    
    for col in cols:
         input_data[col] = input_data[col].fillna(input_data[col].median())
         #Filling Missing Numeric Data with Median
    #<--------------------------------------------------------------------------------->#
    #previously I used--> data[col].fillna(data[col].median(), inplace=True)
    #Error occurs an operation on a copy of a DataFrame or Series, particularly when using the inplace=True argument. 
    #This can lead to unexpected behavior in your code when upgrading pandas versions, especially in pandas 3.0.
    #The specific issue is related to chained assignments, 
    #which happens when you're setting a value on a subset of the DataFrame or Series 
    # (in this case, using data[col].fillna(...) inside a loop). 
    # Pandas warns you about this because the intermediate object data[col] may not directly modify the original data object, 
    #potentially leading to bugs.
    #<--------------------------------------------------------------------------------->#
    
    #I've searched it and found this on Chat GPT 4o
    
    #<--------------------------------------------------------------------------------->#   

    input_data["Embarked"] = input_data["Embarked"].fillna("U")
    #Filling Missing Categorical Data for Embarked, "U" is used as a placeholder meaning "Unknown".
    
    return input_data
    
#<=========================This is a Function End=================================>#
#<================================================================================>#

#applying function to datasets
data = clean(data)
test = clean(test)

In [6]:
data.head(3)
#show only 3 rows of this dataset

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S


In [7]:
#Importing preprocessing module from scikit-learn
from sklearn import preprocessing
#This creates a LabelEncoder object.
le = preprocessing.LabelEncoder()
#The LabelEncoder is used to convert categorical string labels into numeric values, like:
#["male", "female"] → [1, 0]
columns = ["Sex", "Embarked"]
#categorical columns we need to encode.

for col in columns:
    data[col] = le.fit_transform(data[col])
    #Fits the encoder to the data[col] (e.g., "Sex") and transforms it.
    #"female" → 0 "male" → 1
    test[col] = le.transform(test[col])
    #Applies the same encoding (learned from the training data) to the test data.
    print(le.classes_)
    #Prints the mapping of classes that the encoder learned. ( ['female' 'male']['C' 'Q' 'S' 'U'])

    #This code reuses the same encoder (le) for both "Sex" and "Embarked" which technically works
    #but it’s better practice to use separate encoders for each column in real-world projects
    #especially if you save the encodings for future use.
data.head(5)

['female' 'male']
['C' 'Q' 'S' 'U']


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2


In [8]:
#This is the model training preparation stage
#we're about to train a Logistic Regression classifier

from sklearn.linear_model import LogisticRegression
#LogisticRegression: A simple, commonly used classification algorithm that predicts binary outcomes (e.g., survived or not).
from sklearn.model_selection import train_test_split
#train_test_split: A utility function to split your dataset into training and validation sets.

y = data["Survived"]
#y (Target): This is what we're trying to predict → the "Survived" column (1 = survived, 0 = not).
X = data.drop("Survived", axis=1)
#X (Features): These are the columns that help the model make predictions — all columns except "Survived".

#Split into Training and Validation Sets:

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
#train_test_split()-->Randomly splits data into training and validation sets
#test_size=0.2-->20% of the data is used for validation, 80% for training

#random_state=42-->Ensures the same split every time (useful for reproducibility)
#Without random_state, the function randomly picks 20% of data every time we run it.
#This means our model might train on different samples each time, causing different results == hard to debug or compare.
#random_state=42 means The split is fixed.
#that means the code is shareable and it will use the same random_state, other people get the same training/validation split

#X_train, y_train: Used to train the model.
#X_val, y_val: Used to evaluate the model’s performance before submitting predictions on the actual test set.

---
## 🤔 Why we use the number **42** while setting the *random state*?
- It’s just a popular convention in programming culture. (A nod to The Hitchhiker’s Guide to the Galaxy, where “42” is the “answer to life, the universe, and everything.”)

- But we can use **any integer**: 7, 123, 999, whatever — as long as it stays the same for reproducibility.

---

In [11]:
clf = LogisticRegression(random_state=0, max_iter=1000).fit(X_train, y_train)

### CLF

- clf is the classifier object where the trained model is stored. This object now holds all the learned parameters (coefficients) from the training data.
- We can later use clf to make predictions, evaluate the model, or check model parameters.

### .fit(X_train, y_train)
- This is the training step where the model is fitted to the data.
- **X_train**: These are the input features (the data used to make predictions), such as "Age", "Fare", and "Sex" for the Titanic dataset.
- **y_train**: This is the target data (the values you want to predict), which in this case is the "Survived" column (1 = survived, 0 = did not survive).
- **.fit()** tells the model to learn the relationship between the input features (X_train) and the target labels (y_train).

### LogisticRegression()
- This creates an instance of the LogisticRegression model from scikit-learn's linear_model module
- This is a classification algorithm used to predict the probability that an instance belongs to a particular class (e.g., survived or not)

### Parameters
- **random_state=0**: This ensures that the randomness in the model (like how the data is split) is reproducible. Every time we run this code with random_state=0, the results will be the same, which is useful for debugging or comparing results.
- **max_iter=1000**: This specifies the maximum number of iterations the algorithm will run when trying to converge to a solution. Logistic Regression works by iterating to find the best parameters (coefficients) for the model, and if it doesn't converge in the specified number of iterations, it stops.
- 1000 iterations is a safe upper limit for many cases. If you get warnings about convergence, you might want to increase this number.

In [12]:
predictions = clf.predict(X_val)
from sklearn.metrics import accuracy_score
accuracy_score(y_val, predictions)

0.8100558659217877

### Making Predictions with the Model
- **clf.predict(X_val)**: Here, we're using the trained Logistic Regression model (clf) to make predictions on the validation data (X_val).
- **X_val** contains the features (inputs) of the validation set, like "Age", "Fare", etc., for the passengers.
- it generates the predicted labels (survived or not) for each row in X_val. These predictions are stored in the predictions variable.

### predictions
- This will be an array of predicted survival labels (0 or 1)

###  Importing **accuracy_score**
- This function comes from scikit-learn's metrics module.
- It is used to calculate the accuracy of our model by comparing the true values (**y_val**) to the predicted values (**predictions**).

###  Calculating Accuracy: **accuracy_score(y_val, predictions)**
- **y_val**: These are the true labels of the validation set (the actual survival status for the passengers).
- This function compares the true labels (y_val) to the predicted labels (predictions).
- It returns a value between 0 and 1
- *1.0 means perfect accuracy (every prediction is correct)*
- *0.0 means the model made no correct predictions at all.*

## Formula for Accuracy:

Accuracy= 
{(Number of Correct Predictions) / (Total Number of Predictions)}
​


In [15]:
submission_preds = clf.predict(test)
#generating predictions for the test data using the trained Logistic Regression model (clf)
#submission_preds  will now contain an array of predicted labels (0 or 1) for each row in the test dataset.like [1, 0, 0, 1, 1, ...]

In [16]:
#Creating a DataFrame:df = pd.DataFrame(...)
df = pd.DataFrame({"PassengerId": test_ids.values,
                   "Survived": submission_preds,
                  })

- A DataFrame is essentially a 2D table that stores data in rows and columns, similar to a spreadsheet or a SQL table.
- we pass data in a dictionary-like format where the keys are column names and the values are the data for each column.
- **"PassengerId"** is the name of the first column in the DataFrame. This column will contain the Passenger IDs.
- **test_ids.values** is assuming that test_ids is a pandas Series or DataFrame column (e.g., "PassengerId" from the test dataset).
- **.values** converts the pandas Series to a NumPy array. This is useful when we want to pass the data as a plain array to the DataFrame constructor.

- **Survived** is the name of the second column, which will contain the predicted survival labels (0 or 1) for each passenger in the test set.
- **submission_preds** is the array of predicted labels that we generated earlier using the clf.predict(test) method.

After running the code, we’ll have a DataFrame df with two columns:

PassengerId----------------------Survived

892  ------------------------------	0

893  ------------------------------	1

894  ------------------------------	1

895  ------------------------------	0

896  ------------------------------	0

...  ------------------------------	...

# Purpose of This DataFrame:
- The DataFrame **df** is being created to prepare the final results for submission.
- **Kaggle Submission Format**: The Titanic competition requires a submission file with two columns: "PassengerId" and "Survived". This code is constructing that file by combining the predicted results (submission_preds) with the corresponding Passenger IDs from the test set (test_ids).

In [19]:
df.to_csv("titanic_submission.csv", index=False)
#saving the dataframe as a CSV file to submit to kaggle
#index=False ensures that the index (row numbers) are not included in the CSV file, as only the PassengerId and Survived columns are needed.